# Neurodesk for Multiple Sclerosis — a hands-on workshop

**Maintainer:** micmas · **Updated:** June 2026 · **License:** CC BY 4.0

A single, self-contained notebook that loads the neuroimaging tools, downloads an
open MS dataset, and walks through the core MS imaging analyses **inside Neurodesk**:

1. **Set up** — load FSL & FreeSurfer, check the Python stack
2. **Get data** — an openly available MS dataset (co-registered T1 / T2 / FLAIR + expert lesion masks, plus a longitudinal subject)
3. **Look at the data** — what MS lesions look like on FLAIR vs T1
4. **Lesion segmentation** — FreeSurfer **SAMSEG**, scored against the expert mask (Dice)
5. **Brain atrophy** — longitudinal percentage brain volume change with FSL **SIENA**
6. **Going further** — spinal cord (SCT) and quantitative MRI (MTR) on your own data

Run the cells top to bottom. Designed for **[Neurodesk Play](https://play.neurodesk.org/)**, but works in any Neurodesk environment.

### Why Neurodesk for MS?
MS imaging needs reproducible, multi-tool pipelines — lesion segmentation, atrophy, cord imaging, qMRI — often mixing FSL, FreeSurfer, ANTs and more. Neurodesk bundles 90+ tools in a container so the whole analysis runs identically on a laptop, an HPC cluster, or in the browser.

### Dataset & citation
We use the open MS dataset curated in [`muschellij2/open_ms_data`](https://github.com/muschellij2/open_ms_data), which packages the Ljubljana 3D MS lesion benchmark — co-registered, N4-corrected T1/T2/FLAIR with **consensus expert lesion masks** — plus a small longitudinal set.

> Lesjak Ž, et al. *A Novel Public MS Lesion Segmentation Benchmark.* Neuroinformatics 16, 51–63 (2018).
> Renton AI, Dao TT, Johnstone T, et al. *Neurodesk: an accessible, flexible and portable data analysis environment for reproducible neuroimaging.* Nature Methods 21, 804–808 (2024).

## 1. Set up the environment

Neurodesk exposes each neuroimaging package as an **environment module**. The `lmod`
helper loads a module into *this notebook's kernel*, so every later cell — including
`!shell` commands — can see the tool.

In [ ]:
# Load neuroimaging tools into this kernel.
try:
    import lmod
except ImportError:
    import module as lmod   # older Neurodesk images expose it as `module`

# Browse what is installed (uncomment):
# await lmod.avail()

# Load by bare name -> Neurodesk picks the default version.
# If a load errors with an "unknown module/version" message, run
#   await lmod.avail('fsl')
# and pin one, e.g.  await lmod.load('fsl/6.0.7.4')
await lmod.load('fsl')
await lmod.load('freesurfer')
await lmod.list()

In [ ]:
# FSL output format + the Python imaging stack
import os
os.environ['FSLOUTPUTTYPE'] = 'NIFTI_GZ'

import nibabel, numpy as np, matplotlib, nilearn
print('nibabel    ', nibabel.__version__)
print('numpy      ', np.__version__)
print('matplotlib ', matplotlib.__version__)
print('nilearn    ', nilearn.__version__)
%matplotlib inline

In [ ]:
# Confirm the command-line tools actually respond
!flirt -version
!run_samseg --help 2>&1 | head -1
!echo "FREESURFER_HOME=$FREESURFER_HOME"

## 2. Get an open MS dataset

We clone the dataset once into persistent storage. It contains a **cross-sectional**
set (one scan per patient, with an expert lesion mask) and a **longitudinal** set
(multiple timepoints per patient) — enough for both the lesion and atrophy analyses.

> The clone is a few hundred MB and may take a couple of minutes. In a
> storage-constrained session you can sparse-checkout only the `*_resampled`
> folders instead.

In [ ]:
from pathlib import Path

WORK = Path('/neurodesktop-storage/ms-workshop'); WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / 'open_ms_data'

if not DATA.exists():
    !cd {WORK} && git clone --depth 1 https://github.com/muschellij2/open_ms_data.git
else:
    print('Already present:', DATA)
!ls {DATA}

In [ ]:
# Locate the files robustly, without hard-coding deep paths.
def pick_dir(*candidates):
    for c in candidates:
        if Path(c).is_dir():
            return Path(c)
    return None

def grab(folder, *keys):
    """First .nii.gz in `folder` whose name contains all `keys` (case-insensitive)."""
    for f in sorted(Path(folder).glob('*.nii.gz')):
        n = f.name.lower()
        if all(k in n for k in keys):
            return f
    return None

cs = DATA / 'cross_sectional'
cs_root = pick_dir(cs/'coregistered_resampled', cs/'coregistered', cs)   # prefer smaller resampled
patients = sorted(p for p in cs_root.iterdir() if p.is_dir())
print(f'{len(patients)} cross-sectional patients under {cs_root.name}/')

subj = patients[0]
print('Using subject:', subj.name)
print('Files:', [f.name for f in sorted(subj.glob("*.nii.gz"))])

FLAIR = grab(subj, 'flair')
T1    = grab(subj, 't1')
GT    = (grab(subj, 'consensus') or grab(subj, 'gold') or
         grab(subj, 'lesion') or grab(subj, 'gt') or grab(subj, 'mask'))
for label, f in [('FLAIR', FLAIR), ('T1', T1), ('lesion GT', GT)]:
    print(f'{label:10}: {f}')
assert FLAIR and T1, 'FLAIR/T1 not found — check the printed file list and adjust grab() keys.' 

## 3. Look at the data — MS lesions on FLAIR vs T1

On **FLAIR**, white-matter lesions are **bright**; on **T1** the same lesions are
iso- to **hypointense** ("black holes" when chronic). That T1-dark / FLAIR-bright
signature is exactly what the segmentation in the next section keys on.

In [ ]:
from nilearn import plotting

plotting.plot_anat(str(T1),    title=f'{subj.name} — T1')
plotting.plot_anat(str(FLAIR), title=f'{subj.name} — FLAIR (lesions bright)')
if GT:
    plotting.plot_roi(str(GT), bg_img=str(FLAIR),
                      title='Expert consensus lesion mask on FLAIR', cmap='autumn')
plotting.show()

## 4. White-matter lesion segmentation with SAMSEG

**SAMSEG** (FreeSurfer 7+) jointly segments brain structures *and* WM lesions from any
combination of contrasts — no retraining, no GPU. We feed it **T1 (input 0)** and
**FLAIR (input 1)**; `--lesion-mask-pattern 0 1` tells it lesions are *dark in input 0*
and *bright in input 1* — the classic MS appearance.

> Runtime is typically **~10–20 min** on a 1 mm T1+FLAIR pair. `--threads 4` helps.

In [ ]:
SAMSEG = WORK / 'out' / subj.name / 'samseg'
SAMSEG.mkdir(parents=True, exist_ok=True)

!run_samseg --input {T1} {FLAIR} --lesion --lesion-mask-pattern 0 1 --threads 4 --output {SAMSEG}

In [ ]:
# Total lesion volume is reported in samseg.stats
stats_file = SAMSEG / 'samseg.stats'
if stats_file.exists():
    for line in stats_file.read_text().splitlines():
        if 'lesion' in line.lower():
            print(line)
else:
    print('samseg.stats not found — did SAMSEG finish? Check the cell output above.')

In [ ]:
# Pull out the lesion label (SAMSEG uses 99) and score it against the expert mask
!mri_binarize --i {SAMSEG}/seg.mgz --match 99 --o {SAMSEG}/lesions.nii.gz

import nibabel as nib, numpy as np
from nilearn.image import resample_to_img

pred = nib.load(str(SAMSEG / 'lesions.nii.gz'))
pred_d = pred.get_fdata() > 0
print('SAMSEG lesion voxels:', int(pred_d.sum()))

if GT:
    gt = resample_to_img(str(GT), pred, interpolation='nearest')  # match SAMSEG space
    gt_d = gt.get_fdata() > 0
    inter = np.logical_and(pred_d, gt_d).sum()
    denom = pred_d.sum() + gt_d.sum()
    dice = (2.0 * inter / denom) if denom else float('nan')
    print(f'Dice vs expert consensus mask: {dice:.3f}')

In [ ]:
plotting.plot_roi(str(SAMSEG / 'lesions.nii.gz'), bg_img=str(FLAIR),
                  title='SAMSEG lesion segmentation on FLAIR', cmap='autumn')
plotting.show()

### Clinical interpretation
- **Total lesion volume** is a standard MS biomarker, though it correlates only weakly with disability (the *clinico-radiological paradox*).
- **Lesion location** (periventricular, juxtacortical, infratentorial, spinal) carries diagnostic weight under the **McDonald 2017** criteria.
- **Dice ~0.6–0.7** vs a single rater is typical for automated MS tools — inter-rater Dice itself is rarely above ~0.8.
- Other tools to compare: **LST** (SPM), **BIANCA** (FSL), and deep-learning models (nicMSlesions, SAMSEG-DL).

## 5. Longitudinal brain atrophy with SIENA

**SIENA** (FSL) estimates **percentage brain volume change (PBVC)** between two
timepoints — a key MS trial endpoint. Healthy ageing is ≈ −0.2%/yr; MS often
runs ≈ −0.5 to −1.0%/yr.

In [ ]:
lg = DATA / 'longitudinal'
lg_root = pick_dir(lg/'coregistered_resampled', lg/'coregistered', lg)
lg_patients = sorted(p for p in lg_root.iterdir() if p.is_dir())
print(f'{len(lg_patients)} longitudinal patients under {lg_root.name}/')

lsubj = lg_patients[0]
studies = sorted(p for p in lsubj.iterdir() if p.is_dir())
print('Subject', lsubj.name, '— timepoints:', [s.name for s in studies])

# If timepoints are sub-folders, grab a T1 from each; otherwise look for two T1s in the patient folder.
if len(studies) >= 2:
    T1_tp1, T1_tp2 = grab(studies[0], 't1'), grab(studies[1], 't1')
else:
    t1s = sorted(lsubj.rglob('*[tT]1*.nii.gz'))
    T1_tp1, T1_tp2 = (t1s[0], t1s[1]) if len(t1s) >= 2 else (None, None)

print('TP1 T1:', T1_tp1)
print('TP2 T1:', T1_tp2)
assert T1_tp1 and T1_tp2, 'Could not find two T1 timepoints — inspect the longitudinal folder layout.' 

In [ ]:
SIENA = WORK / 'out' / lsubj.name / 'siena'
SIENA.mkdir(parents=True, exist_ok=True)

!siena {T1_tp1} {T1_tp2} -o {SIENA}

In [ ]:
report = SIENA / 'report.siena'
if report.exists():
    txt = report.read_text()
    print(txt)
    for line in txt.splitlines():
        if 'PBVC' in line:
            print('>>>', line.strip())
else:
    print('report.siena not found — check the SIENA output above (and report.html for QC).')

### Clinical interpretation
- **Annualised PBVC** is the headline atrophy metric in MS DMT trials.
- **Pseudoatrophy** early in treatment (resolving inflammatory oedema) can confound short intervals — prefer ≥ 12-month follow-up.
- Single-timepoint alternative: **SIENAX** (normalised brain volume). For regional/cortical detail: the **FreeSurfer longitudinal** stream.

## 6. Going further — spinal cord & quantitative MRI

The open dataset above has no cord or magnetisation-transfer data, so the two cells
below are **guarded**: they run only if you drop your own data at the indicated paths,
and otherwise print a hint. Both are central to MS imaging.

In [ ]:
# --- Spinal cord cross-sectional area (CSA) with SCT ---
CORD_T2 = WORK / 'mydata' / 'cord_T2w.nii.gz'
if CORD_T2.exists():
    try:
        await lmod.load('spinalcordtoolbox')   # if this errors: await lmod.avail('spinal')
    except Exception as e:
        print('Adjust the SCT module name:', e)
    CORD_OUT = WORK / 'out' / 'cord'; CORD_OUT.mkdir(parents=True, exist_ok=True)
    # Newer SCT: `sct_deepseg spinalcord`; older SCT: `sct_deepseg_sc`
    !sct_deepseg_sc -i {CORD_T2} -c t2 -ofolder {CORD_OUT}
    SEG = CORD_OUT / (CORD_T2.stem.replace('.nii','') + '_seg.nii.gz')
    !sct_label_vertebrae -i {CORD_T2} -s {SEG} -c t2 -ofolder {CORD_OUT}
    !sct_process_segmentation -i {SEG} -vert 2:5 -perlevel 1 -o {CORD_OUT}/CSA.csv
    import pandas as pd; display(pd.read_csv(CORD_OUT/'CSA.csv'))
else:
    print('No cord scan found. Drop a cervical T2 at', CORD_T2, 'to run SCT.')
    print('Upper-cervical (C2–C3) CSA is the most replicated cord-atrophy biomarker in MS.')

In [ ]:
# --- Magnetisation Transfer Ratio (MTR): a myelin-sensitive qMRI metric ---
QDIR = WORK / 'mydata' / 'qmri'
mt_on, mt_off = QDIR / 'MTon.nii.gz', QDIR / 'MToff.nii.gz'
if mt_on.exists() and mt_off.exists():
    import nibabel as nib, numpy as np
    on  = nib.load(str(mt_on)).get_fdata()
    off_img = nib.load(str(mt_off)); off = off_img.get_fdata()
    with np.errstate(divide='ignore', invalid='ignore'):
        mtr = np.where(off > 0, 100.0 * (off - on) / off, 0.0)
    QDIR.mkdir(parents=True, exist_ok=True)
    nib.save(nib.Nifti1Image(mtr, off_img.affine, off_img.header), str(QDIR / 'MTR.nii.gz'))
    print('Saved MTR map. Healthy NAWM MTR ~ 30-40%; lesional MTR drops 20-50% (demyelination).')
else:
    print('No MT data found. Add MTon.nii.gz / MToff.nii.gz under', QDIR, 'to compute MTR.')

## Wrap-up & reproducibility

You ran a full MS imaging mini-pipeline in one environment: **lesion segmentation**
scored against an expert mask, and **longitudinal atrophy** — plus hooks for cord and
qMRI work.

To make a run reproducible:
- **Pin the Neurodesk image tag** (e.g. `vnmd/neurodesktop:2026-04-01`, not `latest`).
- **Pin module versions** — replace `await lmod.load('fsl')` with the exact version you used (see `await lmod.list()`).
- Commit this notebook together with those version strings.

**Where to go next:** swap in your own (de-identified) T1+FLAIR, compare SAMSEG against LST or a deep-learning model, or extend the longitudinal analysis to SIENAX / FreeSurfer-longitudinal.